# Regenerate the 3D pass renderings from OMNI — `figs_v2/`

Replaces the three-stage pipeline in `MagCarto_Master_test.py` (OMNI, then a Wind MFI/SWE retry, then a
retry with explicit file paths) with a single pass that uses OMNI only. New renderings go to `figs_v2/` and, with `COPY_TO_REPO`, to the
repository's `figs/`; MagCarto's old `figs/` is not touched.

**What changed from the original, and why**

| | original | here |
|---|---|---|
| upstream parameters | live OMNI fetch, falling back to Wind MFI/SWE for 18 legs | the local OMNI2 cache `omni2_cache.pkl` (`omni_cache` in `local_paths.json`), no fallback |
| Dst | `dst{year}.txt`, missing 2000 and 2002–2004 | OMNI2 `Dst_index`, complete |
| trajectory | live SSCWeb call per leg | local `Wind_Ephemerids.h5`, key `/GSM2`, 10-minute cadence |
| standoff clamp | `delta = clip(delta, 1, 10)` R_E | **removed** — over the 137 rendered legs the standoff spans 1.91–4.99 R_E, except pass 47 inbound (16.85 R_E at M_A = 1.40), the one leg the old cap would have cut |
| Mach number | Alfven from OMNI, magnetosonic on the Wind legs | Alfven throughout, recorded per leg |
| legs without data | silently fell back to Wind | **skipped** and listed |

**No network access.** Everything reads from disk.

Run `build_omni2_cache.ipynb` first if `omni2_cache.pkl` is not there yet.

## 1. Configuration

Every visual property is an explicit argument of the helper that draws that element; the defaults sit
here so they can be changed in one place without touching the drawing code.

In [ ]:
from pathlib import Path
import datetime as dt
import pickle
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from geopack import geopack

import json

# Machine-specific locations come from local_paths.json beside this notebook. That file is not part of the
# repository: copy local_paths.example.json and fill it in. Without it, BOW_SHOCK is the folder above this one.
_local_file = Path("local_paths.json")
LOCAL = json.loads(_local_file.read_text(encoding="utf-8")) if _local_file.exists() else {}
BOW_SHOCK = Path(LOCAL.get("bow_shock", Path.cwd().parent))
MAGCARTO  = BOW_SHOCK / "MagCarto"
OUT_DIR   = MAGCARTO / "figs_v2"

OMNI_CACHE = Path(LOCAL.get("omni_cache", BOW_SHOCK / "OMNI" / "omni2_cache.pkl"))
CROSSINGS  = BOW_SHOCK / "bs_crossings_V02.xlsx"
EPHEM      = BOW_SHOCK / "Bow_Shock_Stats" / "Wind_Ephemerids.h5"
EPHEM_KEY  = "/GSM2"                       # plain column names; "/GSM" is the same data with tuple columns
PARAM_CSV  = MAGCARTO / "pass_parameters_v2.csv"

RE_KM = 6371.0

# --- run control -------------------------------------------------------------------------------
MAX_LEGS         = None    # set to e.g. 3 for a smoke test, None for all 140
OVERWRITE        = False   # False skips a leg whose HTML already exists, so a run can be resumed
ONLY_LEGS        = None    # e.g. [(8, "in"), (12, "in")], as used for release 1.2.1: render only these
                           # (pass, "in"/"out") legs, always overwriting them. None renders every leg
                           # (subject to MAX_LEGS and OVERWRITE). See section 7.
ONLY_LEGS = [(7,"in"), (18,"in"), (20,"in"), (33,"in"), (47,"in"), (53,"out")]
COPY_TO_REPO     = True    # also copy every file this run writes into the repository's figs/
REPO_FIGS        = BOW_SHOCK / "Wind_Bow_Shock_database" / "figs"
ARCHIVE          = BOW_SHOCK / "00_old_versions"   # every file a run replaces is archived here (see its README.md)
OMNI_WINDOW_H    = 4.0     # how far from t0 a complete OMNI hour may be, as in the original pipeline
TRAJECTORY_HOURS = 12.0    # trajectory drawn t0 +- this

# --- model parameters --------------------------------------------------------------------------
GAMMA        = 5.0 / 3.0
BS_ECCENT    = 0.9         # conic eccentricity of the bow shock; < 1 closes far down the tail
N_THETA      = 361
MIN_MACH     = 1.2         # below this the standoff relation runs away; such a leg is skipped
TRACE_RLIM   = 60.0

# --- drawing defaults --------------------------------------------------------------------------
MP_COLOR,  MP_OPACITY,  MP_NAZ  = "#326ba8", 0.35, 100
BS_COLOR,  BS_OPACITY,  BS_NAZ  = "#32a852", 0.20, 80
EARTH_COLORSCALE = [[0, "#1a4d7a"], [0.5, "#2e7cb8"], [1, "#2e7cb8"]]
GRID_COLOR, GRID_WIDTH   = "rgba(255,255,255,0.3)", 1
FL_COLOR,   FL_WIDTH     = "rgba(255,180,60,0.55)", 2
TRAJ_COLOR, TRAJ_WIDTH   = "#d62728", 4
CROSS_COLOR_IN,  CROSS_COLOR_OUT = "#111111", "#8c564b"
CROSS_SIZE       = 5
ANNOT_FONT_SIZE  = 13
FIG_WIDTH, FIG_HEIGHT = 1100, 850

OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in (OMNI_CACHE, CROSSINGS, EPHEM):
    print(f"{'ok ' if p.exists() else 'MISSING':<8} {p}")
print(f"{'->':<8} {OUT_DIR}")
if COPY_TO_REPO:
    print(f"{'ok ' if REPO_FIGS.is_dir() else 'MISSING':<8} {REPO_FIGS}  (copies)")
print(f"{'ok ' if ARCHIVE.is_dir() else 'MISSING':<8} {ARCHIVE}  (old versions)")
print(f"legs to render: {ONLY_LEGS if ONLY_LEGS else 'all'}")

## 2. Load the inputs

In [ ]:
cache = pickle.load(open(OMNI_CACHE, "rb"))
FIELDS = ["Flow_pressure", "Dst_index", "By_GSM", "Bz_GSM", "Alfven_mach_number"]
omni = pd.DataFrame({c: np.asarray(cache[c], dtype=float) for c in FIELDS})
omni["Epoch"] = pd.to_datetime(list(cache["Epoch"]))
omni = omni.dropna(subset=FIELDS).sort_values("Epoch").reset_index(drop=True)
print(f"OMNI: {len(omni)} hours with all five fields valid, "
      f"{omni.Epoch.min()} to {omni.Epoch.max()}")

store = pd.HDFStore(EPHEM, "r")
eph = store.select(EPHEM_KEY)
store.close()
eph.columns = [c[0] if isinstance(c, tuple) else c for c in eph.columns]
eph = eph.sort_index()
print(f"ephemeris: {len(eph)} records, {eph.index.min()} to {eph.index.max()}, "
      f"step {pd.Series(eph.index).diff().median()}")

x = pd.read_excel(CROSSINGS, sheet_name=0)
for c in ("time", "start time", "end time", "start time zoom", "end time zoom"):
    x[c] = pd.to_datetime(x[c])
# A leg is a run of consecutive rows with the same pass and SC direction. Until October 2026 the window
# columns were filled only on the first row of each leg and legs were found from the blank cells; the
# windows are now repeated on every row, so the leg is taken from pass and direction instead. On the
# old file both rules give the same 140 legs.
_key = x[["pass", "SC direction"]]
x["leg"] = (_key != _key.shift()).any(axis=1).cumsum()
assert x.groupby(["pass", "SC direction"])["leg"].nunique().max() == 1, \
    "a (pass, SC direction) pair is split into more than one block of rows"
legs = [(int(g["pass"].iloc[0]), g["SC direction"].iloc[0], g) for _, g in x.groupby("leg")]
print(f"crossing list: {len(x)} crossings in {len(legs)} pass legs")

## 3. Models

Shue et al. (1998) Eqs. 10 and 11 for the magnetopause, and a conic bow shock whose subsolar standoff
follows the gas-dynamic scaling, evaluated with the Alfvén Mach number. `standoff()` is unclamped: over the
137 legs rendered on 1 October 2026 it returns 1.91–4.99 R_E, except pass 47 inbound, where M_A = 1.40
gives 16.85 R_E; the old 1–10 R_E restriction would have cut only that leg. A leg with M_A at or below
`MIN_MACH` is skipped rather than drawn, since the relation diverges as M_A approaches 1.

In [ ]:
def shue98_magnetopause(pdyn_npa, bz_nt, n_theta=N_THETA):
    """Return x, y (R_E) of the magnetopause profile, plus r0 and alpha."""
    r0 = (10.22 + 1.29 * np.tanh(0.184 * (bz_nt + 8.14))) * pdyn_npa ** (-1.0 / 6.6)
    alpha = (0.58 - 0.007 * bz_nt) * (1.0 + 0.024 * np.log(pdyn_npa))
    theta = np.linspace(0.0, np.pi - 1e-6, n_theta)
    r = r0 * (2.0 / (1.0 + np.cos(theta))) ** alpha
    return r * np.cos(theta), r * np.sin(theta), r0, alpha


def standoff(r0_mp, mach, gamma=GAMMA):
    """Gas-dynamic standoff of the shock above the magnetopause nose. Unclamped."""
    return r0_mp * ((gamma - 1.0) * mach**2 + 2.0) / ((gamma + 1.0) * (mach**2 - 1.0))


def bowshock_conic(r0_mp, mach, eccentricity=BS_ECCENT, n_theta=N_THETA):
    """Return x, y (R_E) of the bow shock profile, plus its subsolar distance and the standoff."""
    delta = standoff(r0_mp, mach)
    r_bs0 = r0_mp + delta
    theta = np.linspace(0.0, np.pi - 1e-6, n_theta)
    r = r_bs0 * (1.0 + eccentricity) / (1.0 + eccentricity * np.cos(theta))
    return r * np.cos(theta), r * np.sin(theta), r_bs0, delta


def t96_parmod(pdyn_npa, dst_nt, by_nt, bz_nt):
    parmod = np.zeros(10)
    parmod[0:4] = [pdyn_npa, dst_nt, by_nt, bz_nt]
    return parmod


def trace_field_line(seed_xyz, parmod, ut_seconds, rlim=TRACE_RLIM, r0=1.0):
    """Trace both ways from a seed and return the concatenated line in GSM R_E."""
    geopack.recalc(ut_seconds)
    x0, y0, z0 = seed_xyz
    pieces = []
    for direction in (1, -1):
        _, _, _, xx, yy, zz = geopack.trace(x0, y0, z0, dir=direction, rlim=rlim, r0=r0,
                                            parmod=parmod, exname="t96", inname="igrf")
        pieces.append((np.asarray(xx), np.asarray(yy), np.asarray(zz)))
    (xa, ya, za), (xb, yb, zb) = pieces
    return (np.concatenate([xa[::-1], xb]),
            np.concatenate([ya[::-1], yb]),
            np.concatenate([za[::-1], zb]))


def dipole_seeds(r_inner=2.5, n_lat=6, n_lon=12, tail_radii=(3.5, 5.0), n_tail_angle=6):
    """The same seed set the original notebook used: a shell plus two tailward fans."""
    seeds = []
    for lat in np.linspace(-np.pi / 2, np.pi / 2, n_lat):
        for lon in np.linspace(0, 2 * np.pi, n_lon, endpoint=False):
            seeds.append((r_inner * np.cos(lat) * np.cos(lon),
                          r_inner * np.cos(lat) * np.sin(lon),
                          r_inner * np.sin(lat)))
    for r in tail_radii:
        for angle in np.linspace(0, np.pi / 2, n_tail_angle):
            for sign in (1, -1):
                seeds.append((-r * np.cos(angle), 0.0, sign * r * np.sin(angle)))
    return seeds


print(f"{len(dipole_seeds())} field-line seeds per figure")
_x, _y, _r0, _a = shue98_magnetopause(2.0, -2.0)
print(f"check: Pdyn=2 nPa, Bz=-2 nT -> r0 = {_r0:.2f} R_E, alpha = {_a:.3f}, "
      f"standoff at Ma=8 = {standoff(_r0, 8.0):.2f} R_E")

## 4. Drawing helpers

One helper per drawn element. Colour, opacity, width, font size and the number of azimuths are passed in
at the point of use rather than fixed inside, so a figure can be restyled from the loop below.

In [ ]:
def add_surface_of_revolution(fig, x_profile, y_profile, n_azimuth, color, opacity, name,
                              as_mesh=False, show_legend=True):
    """Revolve a 2D (x, y) profile about the x axis and add it as a surface or a triangulated mesh."""
    phi = np.linspace(0, 2 * np.pi, n_azimuth)
    if as_mesh:
        xs, ys, zs = [], [], []
        for a in phi:
            xs.extend(x_profile); ys.extend(y_profile * np.cos(a)); zs.extend(y_profile * np.sin(a))
        n_r = len(x_profile)
        i, j, k = [], [], []
        for a in range(n_azimuth - 1):
            for b in range(n_r - 1):
                i += [a * n_r + b, (a + 1) * n_r + b]
                j += [(a + 1) * n_r + b, (a + 1) * n_r + (b + 1)]
                k += [a * n_r + (b + 1), a * n_r + (b + 1)]
        fig.add_trace(go.Mesh3d(x=xs, y=ys, z=zs, i=i, j=j, k=k, color=color, opacity=opacity,
                                name=name, showlegend=show_legend, hoverinfo="skip"))
    else:
        xg = np.tile(x_profile, (n_azimuth, 1))
        yg = np.outer(np.cos(phi), y_profile)
        zg = np.outer(np.sin(phi), y_profile)
        fig.add_trace(go.Surface(x=xg, y=yg, z=zg, colorscale=[[0, color], [1, color]],
                                 showscale=False, opacity=opacity, name=name, hoverinfo="skip"))
    return fig


def add_earth(fig, colorscale, n_lon=50, n_lat=30, ambient=0.6, diffuse=0.8, specular=0.3):
    u = np.linspace(0, 2 * np.pi, n_lon)
    v = np.linspace(0, np.pi, n_lat)
    fig.add_trace(go.Surface(x=np.outer(np.cos(u), np.sin(v)), y=np.outer(np.sin(u), np.sin(v)),
                             z=np.outer(np.ones_like(u), np.cos(v)), colorscale=colorscale,
                             showscale=False, name="Earth", hoverinfo="skip",
                             lighting=dict(ambient=ambient, diffuse=diffuse, specular=specular)))
    return fig


def add_latitude_grid(fig, color, width, n_lat=7, n_lon=100):
    for lat in np.linspace(-np.pi / 2, np.pi / 2, n_lat):
        lons = np.linspace(0, 2 * np.pi, n_lon)
        fig.add_trace(go.Scatter3d(x=np.cos(lat) * np.cos(lons), y=np.cos(lat) * np.sin(lons),
                                   z=np.sin(lat) * np.ones_like(lons), mode="lines",
                                   line=dict(color=color, width=width),
                                   showlegend=False, hoverinfo="skip"))
    return fig


def add_field_lines(fig, lines, color, width, name="Field lines"):
    for n, (X, Y, Z) in enumerate(lines):
        fig.add_trace(go.Scatter3d(x=X, y=Y, z=Z, mode="lines",
                                   line=dict(color=color, width=width),
                                   name=name, showlegend=(n == 0), hoverinfo="skip"))
    return fig


def add_trajectory(fig, xyz_re, times, color, width, name="Wind trajectory"):
    fig.add_trace(go.Scatter3d(x=xyz_re[:, 0], y=xyz_re[:, 1], z=xyz_re[:, 2], mode="lines",
                               line=dict(color=color, width=width), name=name,
                               text=[t.strftime("%Y-%m-%d %H:%M") for t in times],
                               hovertemplate="%{text}<br>X=%{x:.1f} Y=%{y:.1f} Z=%{z:.1f} Re<extra></extra>"))
    return fig


def add_crossing_markers(fig, xyz_re, labels, color, size, name):
    if not len(xyz_re):
        return fig
    fig.add_trace(go.Scatter3d(x=xyz_re[:, 0], y=xyz_re[:, 1], z=xyz_re[:, 2], mode="markers",
                               marker=dict(color=color, size=size, line=dict(color="white", width=1)),
                               name=name, text=labels,
                               hovertemplate="%{text}<br>X=%{x:.1f} Y=%{y:.1f} Z=%{z:.1f} Re<extra></extra>"))
    return fig


def set_layout(fig, title, annotation, font_size, width, height, axis_range):
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor="left"),
        width=width, height=height, showlegend=True,
        scene=dict(xaxis=dict(title="X GSM [Re]", range=axis_range),
                   yaxis=dict(title="Y GSM [Re]", range=axis_range),
                   zaxis=dict(title="Z GSM [Re]", range=axis_range),
                   aspectmode="cube"),
        annotations=[dict(text=annotation, showarrow=False, xref="paper", yref="paper",
                          x=0.0, y=0.0, align="left", font=dict(size=font_size, family="monospace"))],
        margin=dict(l=0, r=0, t=60, b=90))
    return fig

## 5. Per-leg data assembly

`omni_at()` mirrors the original pipeline's progressive window, but over hours where all five fields are
valid together, and it reports how far away the hour it used was.

In [ ]:
_omni_epoch = omni.Epoch.values

def omni_at(t0, window_h=OMNI_WINDOW_H):
    """Nearest hour with all five fields valid, or None if none within window_h."""
    k = np.searchsorted(_omni_epoch, np.datetime64(t0))
    best = None
    for i in (k - 1, k):
        if 0 <= i < len(omni):
            off = (omni.Epoch[i] - t0).total_seconds() / 3600.0
            if best is None or abs(off) < abs(best[1]):
                best = (i, off)
    if best is None or abs(best[1]) > window_h:
        return None
    i, off = best
    r = omni.loc[i]
    return dict(Pdyn=float(r.Flow_pressure), Dst=float(r.Dst_index), By=float(r.By_GSM),
                Bz=float(r.Bz_GSM), Ma=float(r.Alfven_mach_number),
                omni_epoch=r.Epoch, offset_h=off)


def reference_time(group):
    """The original used the centre of the first zoom window, else the first crossing."""
    z = group.dropna(subset=["start time zoom", "end time zoom"])
    if len(z):
        a = z["start time zoom"].iloc[0]
        b = z["end time zoom"].iloc[0]
        return a + (b - a) / 2
    return group["time"].min()


def trajectory(t0, hours=TRAJECTORY_HOURS):
    w = eph.loc[t0 - pd.Timedelta(hours=hours): t0 + pd.Timedelta(hours=hours)]
    xyz = w[["GSM_X [km]", "GSM_Y [km]", "GSM_Z [km]"]].to_numpy() / RE_KM
    return w.index, xyz


def positions_at(times):
    idx = eph.index.get_indexer(pd.DatetimeIndex(times), method="nearest")
    return eph.iloc[idx][["GSM_X [km]", "GSM_Y [km]", "GSM_Z [km]"]].to_numpy() / RE_KM


# smoke test on the first leg
_pid, _dir, _g = legs[0]
_t0 = reference_time(_g)
print(f"leg 1 = pass {_pid} {_dir}, t0 = {_t0}")
print("omni:", omni_at(_t0))
_ti, _xyz = trajectory(_t0)
print(f"trajectory points: {len(_ti)}, first {_xyz[0].round(1)} Re")

## 6. Build one figure

Returns the figure and the record of parameters that went into it.

In [ ]:
def build_figure(pass_id, sc_direction, group):
    t0 = reference_time(group)
    rec = dict(pass_id=pass_id, sc_direction=sc_direction, t0=t0,
               n_crossings=len(group), status="", html="")

    o = omni_at(t0)
    if o is None:
        rec["status"] = f"skipped: no complete OMNI hour within {OMNI_WINDOW_H} h"
        return None, rec
    rec.update({k: o[k] for k in ("Pdyn", "Dst", "By", "Bz", "Ma", "offset_h")})
    rec["omni_epoch"] = o["omni_epoch"]

    if o["Ma"] <= MIN_MACH:
        rec["status"] = f"skipped: Ma = {o['Ma']:.2f} <= {MIN_MACH}, standoff relation diverges"
        return None, rec

    xmp, ymp, r0_mp, alpha = shue98_magnetopause(o["Pdyn"], o["Bz"])
    xbs, ybs, r_bs0, delta = bowshock_conic(r0_mp, o["Ma"])
    rec.update(r0_mp=r0_mp, alpha=alpha, r_bs0=r_bs0, delta=delta, clamped=False)

    parmod = t96_parmod(o["Pdyn"], o["Dst"], o["By"], o["Bz"])
    ut = (t0 - dt.datetime(1970, 1, 1)).total_seconds()
    lines = []
    for seed in dipole_seeds():
        try:
            lines.append(trace_field_line(seed, parmod, ut))
        except Exception:
            pass
    rec["n_field_lines"] = len(lines)

    times, xyz = trajectory(t0)
    inb = group[group["direction"] == "in"]["time"]
    outb = group[group["direction"] == "out"]["time"]

    span = max(np.abs(xyz).max() if len(xyz) else 0.0, r_bs0 * 1.2, 40.0)
    axis_range = [-span, span]

    fig = go.Figure()
    add_surface_of_revolution(fig, xmp, ymp, MP_NAZ, MP_COLOR, MP_OPACITY,
                              name="Magnetopause (Shue 1998)", as_mesh=True)
    add_surface_of_revolution(fig, xbs, ybs, BS_NAZ, BS_COLOR, BS_OPACITY,
                              name="Bow shock", as_mesh=False)
    add_earth(fig, EARTH_COLORSCALE)
    add_latitude_grid(fig, GRID_COLOR, GRID_WIDTH)
    add_field_lines(fig, lines, FL_COLOR, FL_WIDTH)
    add_trajectory(fig, xyz, times, TRAJ_COLOR, TRAJ_WIDTH)
    add_crossing_markers(fig, positions_at(inb), [t.strftime("in  %H:%M:%S") for t in inb],
                         CROSS_COLOR_IN, CROSS_SIZE, "Crossing inward")
    add_crossing_markers(fig, positions_at(outb), [t.strftime("out %H:%M:%S") for t in outb],
                         CROSS_COLOR_OUT, CROSS_SIZE, "Crossing outward")

    title = f"Pass {pass_id:02d} {sc_direction}bound &mdash; {t0:%Y-%m-%d %H:%M} UT"
    annot = (f"OMNI2 {o['omni_epoch']:%Y-%m-%d %H:%M} ({o['offset_h']:+.1f} h)   "
             f"Pdyn={o['Pdyn']:.2f} nPa   Bz={o['Bz']:+.2f} nT   By={o['By']:+.2f} nT   "
             f"Dst={o['Dst']:.0f} nT   M_A={o['Ma']:.1f}<br>"
             f"Shue r0={r0_mp:.2f} Re   alpha={alpha:.3f}   "
             f"bow shock standoff={delta:.2f} Re (unclamped)   subsolar={r_bs0:.2f} Re   "
             f"e={BS_ECCENT}   {len(lines)} field lines")
    set_layout(fig, title, annot, ANNOT_FONT_SIZE, FIG_WIDTH, FIG_HEIGHT, axis_range)

    rec["status"] = "ok"
    return fig, rec

## 7. Run

Three ways to run this cell:

- **Selected legs.** List them in `ONLY_LEGS` in cell 1 as `(pass, "in")` or `(pass, "out")`. Only those
  legs are rendered, and they are always overwritten. Release 1.2.1 used this for pass 8 inbound, whose five
  crossings had had no direction, and pass 12 inbound, whose zoom window had carried the wrong date.
- **Smoke test.** `ONLY_LEGS = None` and `MAX_LEGS = 2`; prints the per-leg time so a full run can be
  estimated.
- **Everything.** `ONLY_LEGS = None` and `MAX_LEGS = None`. With `OVERWRITE = False` the loop can be
  interrupted and resumed.

Every file a run replaces is archived, never deleted: a rendering of the same leg in `figs_v2/` or in the
repository's `figs/` (same name or an older timestamp), and the previous `pass_parameters_v2.csv`. Each goes
to `bow_shock/00_old_versions/<date>_<time>_render_passes_v2/` under its original path and gets a row in
`00_old_versions/INDEX.csv`. With `COPY_TO_REPO = True` each new rendering is also copied into the
repository's `figs/`.

In [ ]:
import csv
import hashlib
import shutil


def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def archive_file(path, event, replaced_by, copy=False, note=""):
    """Move (or copy) path to ARCHIVE/event/<path relative to BOW_SHOCK> and add a row to INDEX.csv.
    Never overwrites. The naming rules are in ARCHIVE/README.md."""
    path = Path(path)
    rel = path.relative_to(BOW_SHOCK)
    dest = ARCHIVE / event / rel
    if dest.exists():
        raise FileExistsError(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    digest, size = _md5(path), path.stat().st_size
    modified = dt.datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
    (shutil.copy2 if copy else shutil.move)(str(path), str(dest))
    assert _md5(dest) == digest, f"archived copy differs from the original: {dest}"
    index = ARCHIVE / "INDEX.csv"
    new_index = not index.exists()
    with open(index, "a", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        if new_index:
            w.writerow(["event", "archived_path", "original_location", "file", "version_modified",
                        "archived_on", "replaced_by", "md5", "bytes", "note"])
        w.writerow([event, str(Path(event) / rel), str(rel), path.name, modified,
                    dt.datetime.now().strftime("%Y-%m-%d %H:%M"), replaced_by, digest, size, note])
    return dest


def new_event(name):
    """A folder name for this run that is not in use yet: YYYY-MM-DD_HHMM_<name>, then -2, -3 ... if needed."""
    base = f"{dt.datetime.now():%Y-%m-%d_%H%M}_{name}"
    event, k = base, 1
    while (ARCHIVE / event).exists():
        k += 1
        event = f"{base}-{k}"
    return event


RUN_EVENT = new_event("render_passes_v2")    # this run's folder in ARCHIVE


def leg_files(folder, pass_id, sc_direction):
    """Every rendering of this leg in folder, whatever its timestamp."""
    return sorted(Path(folder).glob(
        f"Magnetosphere_Pass_{pass_id:02d}_{sc_direction.capitalize()}bound_*.html"))


if ONLY_LEGS:
    wanted = [(int(p), str(d)) for p, d in ONLY_LEGS]
    by_key = {(p, d): (p, d, g) for p, d, g in legs}
    missing = [k for k in wanted if k not in by_key]
    if missing:
        raise ValueError(f"not in the crossing list: {missing}")
    todo, force = [by_key[k] for k in wanted], True
else:
    todo = legs if MAX_LEGS is None else legs[:MAX_LEGS]
    force = OVERWRITE

records = []
t_start = time.time()

for n, (pass_id, sc_direction, group) in enumerate(todo, 1):
    t0 = reference_time(group)
    name = f"Magnetosphere_Pass_{pass_id:02d}_{sc_direction.capitalize()}bound_{t0:%Y-%m-%dT%H%M}.html"
    target = OUT_DIR / name
    if target.exists() and not force:
        print(f"[{n}/{len(todo)}] pass {pass_id:02d} {sc_direction}: exists, skipping")
        continue

    t_leg = time.time()
    try:
        fig, rec = build_figure(pass_id, sc_direction, group)
    except Exception as e:
        rec = dict(pass_id=pass_id, sc_direction=sc_direction, t0=t0, status=f"error: {e}")
        fig = None
    if fig is not None:
        for f in leg_files(OUT_DIR, pass_id, sc_direction):
            archive_file(f, RUN_EVENT, "new rendering of this leg")
            print(f"      archived {f.name}")
        fig.write_html(str(target), include_plotlyjs="cdn", full_html=True)
        rec["html"] = name
        if COPY_TO_REPO and REPO_FIGS.is_dir():
            for f in leg_files(REPO_FIGS, pass_id, sc_direction):
                archive_file(f, RUN_EVENT, "new rendering of this leg")
                print(f"      archived {f.name}  (repository copy)")
            shutil.copy2(target, REPO_FIGS / name)
    rec["seconds"] = round(time.time() - t_leg, 1)
    records.append(rec)
    print(f"[{n}/{len(todo)}] pass {pass_id:02d} {sc_direction:<3} {rec['status']:<60} "
          f"{rec['seconds']:6.1f} s")

print(f"\ntotal {time.time() - t_start:.0f} s for {len(records)} legs")

## 8. Parameter table and summary

`pass_parameters_v2.csv` is the table Section 2 of the paper needs: what upstream parameters each
rendering used, how far from the crossing they came from, and what standoff distance followed.

The table is updated, not rewritten: rows for the legs rendered in this run replace their old rows, every
other row is kept, and the previous table is archived in `00_old_versions` first. A run that rendered no
leg leaves the table alone. The summary
below covers the whole table.

In [ ]:
import shutil

COLS = ["pass_id", "sc_direction", "t0", "n_crossings", "status", "omni_epoch", "offset_h", "Pdyn", "Bz",
        "By", "Dst", "Ma", "r0_mp", "alpha", "delta", "r_bs0", "clamped", "n_field_lines", "seconds", "html"]

new = pd.DataFrame(records)
if new.empty:
    par = pd.read_csv(PARAM_CSV, float_precision="round_trip")
    print(f"no leg rendered in this run; {PARAM_CSV.name} left as it was\n")
else:
    if PARAM_CSV.exists():
        archive_file(PARAM_CSV, RUN_EVENT, "rows of the legs rendered in this run replaced", copy=True)
        old = pd.read_csv(PARAM_CSV, float_precision="round_trip")   # keeps every untouched row byte-identical
        done = set(zip(new.get("pass_id", []), new.get("sc_direction", [])))
        keep = old[[(p, d) not in done for p, d in zip(old.pass_id, old.sc_direction)]]
        par = pd.concat([keep, new], ignore_index=True)
    else:
        par = new
    order = {(p, d): i for i, (p, d, _) in enumerate(legs)}
    par["_order"] = [order.get((int(p), d), len(order)) for p, d in zip(par.pass_id, par.sc_direction)]
    par = par.sort_values("_order", kind="stable").drop(columns="_order").reset_index(drop=True)
    par["t0"] = pd.to_datetime(par["t0"]).dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]
    if "omni_epoch" in par:
        par["omni_epoch"] = pd.to_datetime(par["omni_epoch"]).dt.strftime("%Y-%m-%d %H:%M:%S")
    par = par[[c for c in COLS if c in par.columns]]
    par.to_csv(PARAM_CSV, index=False)
    print(f"wrote {PARAM_CSV}: {len(par)} legs, {len(new)} from this run\n")
    if len(new):
        show = [c for c in ["pass_id", "sc_direction", "t0", "status", "omni_epoch", "offset_h", "Ma", "delta", "html"]
                if c in new.columns]
        print(new[show].to_string(index=False), "\n")

ok = par[par.status == "ok"] if "status" in par else par
print(f"rendered: {len(ok)}   skipped or failed: {len(par) - len(ok)}")
if len(par) - len(ok):
    print(par[par.status != "ok"][["pass_id", "sc_direction", "t0", "status"]].to_string(index=False))

if len(ok):
    print(f"\nOMNI offset  : median {ok.offset_h.abs().median():.2f} h   max {ok.offset_h.abs().max():.2f} h")
    print(f"M_A          : min {ok.Ma.min():.2f}   median {ok.Ma.median():.1f}   max {ok.Ma.max():.1f}")
    print(f"Shue r0      : min {ok.r0_mp.min():.2f}   median {ok.r0_mp.median():.2f}   max {ok.r0_mp.max():.2f} Re")
    print(f"standoff     : min {ok.delta.min():.2f}   median {ok.delta.median():.2f}   max {ok.delta.max():.2f} Re")
    print(f"field lines  : min {ok.n_field_lines.min()}   median {ok.n_field_lines.median():.0f} of 96")
    print(f"\nSentence for Section 2: the standoff distance ranges from {ok.delta.min():.1f} to "
          f"{ok.delta.max():.1f} Re across the {len(ok)} rendered passes, with no limiting applied.")

## Expected outcome

Full run, as recorded in `pass_parameters_v2.csv` by the run of 1 October 2026:

- **3 legs skipped**: passes 4, 55 and 56 inbound (11 crossings), with no hour within 4 h of t0 at which
  all five OMNI fields are present.
- **137 rendered** (624 crossings then; 633 after the list corrections of 8 October 2026), 132 of them from an OMNI hour within 1 h of t0; the largest offset is
  3.19 h.
- M_A from 1.40 to 39.70, Shue r0 from 7.42 to 13.62 R_E, standoff from 1.91 to 16.85 R_E (above 5 R_E only
  for pass 47 inbound).
- 96 field lines in every rendering; no leg reports `clamped`.

Release 1.2.1 re-rendered passes 8 and 12 inbound with `ONLY_LEGS = [(8, "in"), (12, "in")]` on 8 October 2026.
Pass 8 inbound kept its file name and gained its five crossing markers. Pass 12 inbound became
`Magnetosphere_Pass_12_Inbound_1996-10-04T0055.html`, with the OMNI2 hour 1996-10-04 00:30 (M_A = 9.6).
The files they replaced are in `00_old_versions/2026-10-08_1346_render_passes_v2/`, and every other row of
`pass_parameters_v2.csv` stayed as it was.

After the six legs were corrected on 8 October 2026, release 1.2.1 re-renders them with
`ONLY_LEGS = [(7, "in"), (18, "in"), (20, "in"), (33, "in"), (47, "in"), (53, "out")]`. Their windows did
not change, so their file names and OMNI2 hours stay the same. Only their crossing markers change, and in
`pass_parameters_v2.csv` their `n_crossings` (passes 33 and 47 inbound and 53 outbound) and `seconds`.

If the numbers differ from these, the OMNI cache or the reference-time rule has changed and it is worth
stopping to find out why before the figures go into the paper.